In [1]:
# # Upgrade langchain to latest version
# !pip install --upgrade langchain

# # Install chromadb (required for Chroma vectorstore)
# !pip install chromadb

#!pip install langchain-huggingface


In [2]:
#pip install langchain-chroma


In [3]:
import sys
print(sys.executable)

/home/intern5/PythonProgramming/myenv/venv/bin/python3


In [4]:
#!{sys.executable} -m pip install --upgrade langchain chromadb

In [5]:
import langchain
import chromadb

# Check version
print("LangChain version:", langchain.__version__)



LangChain version: 1.2.13


In [6]:
# File handling
import os

# LangChain imports
import glob

# LangChain imports
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document # Updated import for Document

from dotenv import load_dotenv

# 1. Load the specific file
load_dotenv(dotenv_path="hugface.env")

# 2. Get the key using your specific name
hf_token = os.getenv("HUGGING_FACE_API_KEY")

# 3. Initialize the embeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
    model_kwargs={'token': hf_token}
)


/home/intern5/PythonProgramming/myenv/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 4494.29it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
documents = []
filenames = []

for file in glob.glob("TextualData/*.txt"):  # adjust your folder path
    with open(file, "r", encoding="utf-8") as f:
        documents.append(f.read())
        filenames.append(file)  # store filename

# Now you can print them
for i, (filename, content) in enumerate(zip(filenames, documents)):
    print(f"Document {i}: {filename}")
    print(content[:200])  # preview first 200 characters
    print("---")

Document 0: TextualData/Microsoft.txt
﻿Microsoft
Microsoft Corporation is an American multinational Microsoft Corporation
corporation and technology conglomerate
headquartered in Redmond, Washington.[2] Founded
in 1975, the company became
---
Document 1: TextualData/Google.txt
﻿Google
Google LLC (/ˈɡuːɡəl/ ⓘ , GOO-gəl) is an Google LLC
American multinational corporation and technology
company focusing on online advertising, search engine
technology, cloud computing, compute
---
Document 2: TextualData/SpaceX.txt
﻿SpaceX
Space Exploration Technologies Corp., commonly Space Exploration Technologies
referred to as SpaceX, is an American space Corp.
technology company headquartered at the Starbase
development sit
---
Document 3: TextualData/Nvidia.txt
﻿Nvidia
Nvidia Corporation[a] (/ɛnˈvɪdiə/ en-VID-ee-ə) is an Nvidia Corporation
American technology company headquartered in
Santa Clara, California. Founded in 1993 by Jensen
Huang (president and CEO
---
Document 4: TextualData/Tesla.txt
﻿Te

In [14]:
import os
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# 1. Get the FULL path to your current folder
current_dir = os.getcwd() 
persistent_directory = os.path.join(current_dir, "TextualData")

print(f"Looking for database at: {persistent_directory}")

# 2. Check if the folder actually exists before trying to load it
if not os.path.exists(persistent_directory):
    print(f" ERROR: The folder '{persistent_directory}' was not found!")
    print("You need to run your 'Create Database' script first.")
else:
    # 3. Load the embeddings
    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2") 

    # 4. Load the database
    db = Chroma(
        persist_directory=persistent_directory,   
        embedding_function=embeddings,
        collection_metadata={"hnsw:space": "cosine"}  
    )

    # 5. Retrieve
    retriever = db.as_retriever(search_kwargs={"k": 5})
    query = "How much did Microsoft pay to acquire GitHub?"
    
    relevant_docs = retriever.invoke(query)
    
    if len(relevant_docs) > 0:
        print(f"Success! Found {len(relevant_docs)} documents.")
        print(relevant_docs[0].page_content[:200])
    else:
        print("Database found, but it is empty. Did you add documents to it?")


Looking for database at: /home/intern5/PythonProgramming/myenv/TextualData


Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 4772.99it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Database found, but it is empty. Did you add documents to it?


In [11]:
print(f"Number of documents found: {len(relevant_docs)}")

if not relevant_docs:
    print("No relevant documents were found. Check your search query or database content.")
else:
    print("--- Retrieved Context ---")
    for i, doc in enumerate(relevant_docs, 1):
        print(f"\nDocument {i}:")
        print(doc.page_content)


Number of documents found: 0
No relevant documents were found. Check your search query or database content.


In [ ]:
print("")